In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuration
DATA_DIR = "lightsource"

FILES = {
    'transformers_original': 'transformers_original.csv',
    'transformers_updated': 'transformers_updated.csv',
    'lines_original': 'lines_original.csv',
    'lines_updated': 'lines_updated.csv'
}

COLUMNS_TO_ANALYZE = {
    'transformers': ['b', 'x', 'r'],
    'lines': ['b', 'x', 'r']
}

IDENTIFIER_COLUMNS = {
    'transformers': 'Transformer',
    'lines': ['bus0', 'bus1', 'type']
}

# Create output directory
OUTPUT_DIR_BASE = os.path.join(DATA_DIR, "analysis_results")
os.makedirs(OUTPUT_DIR_BASE, exist_ok=True)


def load_csv(file_path):
    """Load a CSV file into a pandas DataFrame."""
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded {file_path} successfully.")
        return df
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None


def plot_corresponding_values(original_df, updated_df, component, column_name):
    """Plot corresponding values for original and updated datasets, excluding identical values."""
    identifier = IDENTIFIER_COLUMNS.get(component)
    output_dir = os.path.join(OUTPUT_DIR_BASE, column_name)
    os.makedirs(output_dir, exist_ok=True)

    if isinstance(identifier, list):
        merged_df = pd.merge(original_df, updated_df, on=identifier, suffixes=('_original', '_updated'))
    else:
        merged_df = pd.merge(original_df, updated_df, on=identifier, suffixes=('_original', '_updated'))

    col_original = f'{column_name}_original'
    col_updated = f'{column_name}_updated'

    # Convert to numeric and remove NaN values
    merged_df[col_original] = pd.to_numeric(merged_df[col_original], errors='coerce')
    merged_df[col_updated] = pd.to_numeric(merged_df[col_updated], errors='coerce')
    merged_df = merged_df.dropna(subset=[col_original, col_updated])

    # **Remove exact identical values**
    different_values = merged_df[merged_df[col_original] != merged_df[col_updated]]

    if different_values.empty:
        print(f"No different values found for {component} {column_name}. Skipping plot.")
        return

    plt.figure(figsize=(8, 8))

    # Scatter plot with a diagonal reference line
    plt.scatter(different_values[col_original], different_values[col_updated], alpha=0.6)
    min_val, max_val = different_values[[col_original, col_updated]].min().min(), different_values[[col_original, col_updated]].max().max()
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='y = x')

    plt.xlabel(f'Original {column_name}')
    plt.ylabel(f'Updated {column_name}')
    plt.title(f'Changed {column_name} Values for {component.capitalize()} (Excluding Identical Values)')
    plt.legend()
    plt.tight_layout()

    # Save plot
    plot_path = os.path.join(output_dir, f"{component}_corresponding_{column_name}_plot.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"Scatter plot saved: {plot_path}")


def plot_histogram_different_values(original, updated, component, column_name):
    """Create and save overlapping histograms for original and updated values, excluding identical values."""
    output_dir = os.path.join(OUTPUT_DIR_BASE, column_name)
    os.makedirs(output_dir, exist_ok=True)

    original = pd.Series(original).dropna()
    updated = pd.Series(updated).dropna()

    # **Remove identical values before plotting histograms**
    mask = original != updated
    original_diff = original[mask]
    updated_diff = updated[mask]

    if len(original_diff) == 0:
        print(f"No different values found for {component} {column_name} histogram. Skipping.")
        return

    plt.figure(figsize=(10, 6))
    sns.histplot(original_diff, color='blue', label='Original', kde=True, alpha=0.6)
    sns.histplot(updated_diff, color='orange', label='Updated', kde=True, alpha=0.6)
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.title(f"Distribution of Different {column_name} Values for {component.capitalize()}")
    plt.legend()
    plt.tight_layout()

    # Save plot
    plot_path = os.path.join(output_dir, f"{component}_{column_name}_histogram.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"Histogram saved: {plot_path}")


def main():
    print("Starting histogram and scatter plot analysis...")

    # Load all datasets
    datasets = {key: load_csv(os.path.join(DATA_DIR, filename)) for key, filename in FILES.items()}

    # Process each component and column
    for component in ['transformers', 'lines']:
        for column_name in COLUMNS_TO_ANALYZE[component]:
            print(f"\nProcessing {column_name} for {component}...")

            original_key = f"{component}_original"
            updated_key = f"{component}_updated"

            if original_key in datasets and updated_key in datasets:
                original_df = datasets[original_key]
                updated_df = datasets[updated_key]

                if column_name in original_df.columns and column_name in updated_df.columns:
                    original_values = pd.to_numeric(original_df[column_name], errors='coerce').dropna()
                    updated_values = pd.to_numeric(updated_df[column_name], errors='coerce').dropna()

                    # Generate scatter plots and histograms (excluding identical values)
                    plot_corresponding_values(original_df, updated_df, component, column_name)
                    plot_histogram_different_values(original_values, updated_values, component, column_name)
                else:
                    print(f"Column {column_name} not found in one of the datasets for {component}. Skipping.")

    print("\nAnalysis completed.")


if __name__ == "__main__":
    main()


Starting histogram and scatter plot analysis...
Loaded lightsource/transformers_original.csv successfully.
Loaded lightsource/transformers_updated.csv successfully.
Loaded lightsource/lines_original.csv successfully.
Loaded lightsource/lines_updated.csv successfully.

Processing b for transformers...
Scatter plot saved: lightsource/analysis_results/b/transformers_corresponding_b_plot.png
Histogram saved: lightsource/analysis_results/b/transformers_b_histogram.png

Processing x for transformers...
Scatter plot saved: lightsource/analysis_results/x/transformers_corresponding_x_plot.png
Histogram saved: lightsource/analysis_results/x/transformers_x_histogram.png

Processing r for transformers...
Scatter plot saved: lightsource/analysis_results/r/transformers_corresponding_r_plot.png
Histogram saved: lightsource/analysis_results/r/transformers_r_histogram.png

Processing b for lines...


KeyError: 'bus0'